In [90]:
# Imports
import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[2]))

import os

import torch
from attacks.fl.models import OBU
from attacks.fl.utils import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

In [122]:
# Inputs
checkpoint_file="../../../saved_models/RandomPos-cenFL-norm.ckpt"
data_filename = "RandomPos_0709.csv"
train_perc = 80
norm_trained = True


In [ ]:
# Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        # ART passes one-hot labels; CrossEntropyLoss needs class indices
        if b.dim() == 3:
            b = b.argmax(dim=-1)  # (batch, seq_len, num_classes) → (batch, seq_len)
        return self.loss(
            a.permute(0, 2, 1),  # (batch, seq_len, C) → (batch, C, seq_len)
            b.long()
        )

class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        
        if not norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):
        if not norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)
        return logits

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [4]:
# Load data (very time consuming part)
data_file = f"../../../data/{data_filename}"
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, train_perc)


In [87]:
# Set model variables
wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                     data_min=x_train.min(axis=(0,1)),
                                     data_max=x_train.max(axis=(0,1)))
criterion = SequenceCrossEntropy()
optimizer = optim.Adam(
    wrapped_model.parameters(),
    lr=0.001
)

classifier = PyTorchClassifier(
    model=wrapped_model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(10, 7),
    nb_classes=20,
    clip_values=(0.0, 1.0), # for normalized
    device_type="cpu"
)

print(type(model))
print(type(model.model))
print(type(wrapped_model))

<class 'attacks.fl.models.OBU'>
<class 'attacks.fl.models.Modena'>
<class '__main__.NormalizedCfCWrapper'>


In [ ]:
# REGULAR: Test the model
out = model.test(x_test, y_test, mathy=True)
print(f"Out: {out}")

torch.Size([125094, 10, 20])
0.9641379419885481
0.978858930529575
Model got 1229527/1250940 right.
Accuracy: 0.9828824723807696, Precision: 0.9641379419885481, Recall: 0.978858930529575, F1 Score: 0.9714426699563232
878868, 70.2566070315123% Zeroes, 372072 Non Zero entries.
Out: (0.9714426699563232, 0.978858930529575, 0.9641379419885481, 0.9828824723807696)


In [ ]:
# Normalize data before giving it into ART
eps = 1e-8 # Approximate 0

# x values
data_min_x = x_train.min(axis=(0, 1))  # shape (7,)
data_max_y = x_train.max(axis=(0, 1))  # shape (7,)

denom_x = data_max_y - data_min_x
denom_x_safe = np.where (denom_x == 0, eps, denom_x) # Need to account for divide by 0 cases
x_train_norm = (x_train - data_min_x) / denom_x_safe
x_test_norm = (x_test - data_min_x) / denom_x_safe

x_reconstructed = x_test_norm * denom_x_safe + data_min_x

Sanity check, should be ~0:  6.1035156e-05


In [98]:
# Normalization checks
print("Sanity check, should be ~0: ", np.abs(x_reconstructed - x_test).max())  # should be ~0 (floating point noise only)
print("x nan/inf:", np.isnan(x_test_norm).any(), np.isinf(x_test_norm).any())
print("y nan/inf:", np.isnan(y_test).any(), np.isinf(y_test).any())
print("y unique:", np.unique(y_test))

Sanity check, should be ~0:  6.1035156e-05
x nan/inf: False False
y nan/inf: False False
y unique: [0 3]


In [94]:
# Benign test

benign_predictions = classifier.predict(x_test_norm, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
accuracy = np.sum(benign_pred_classes == y_test) / benign_pred_classes.size
print(f"Benign accuracy: {accuracy}")


true_flat = y_test.flatten()
pred_flat = benign_pred_classes.flatten()
print("Precision:", precision_score(true_flat, pred_flat, average="weighted", zero_division=0))
print("Recall:", recall_score(true_flat, pred_flat, average="weighted", zero_division=0))
print("F1:", f1_score(true_flat, pred_flat, average="weighted", zero_division=0))

# 98.29% accuracy

Benign accuracy: 0.9828824723807696
Precision: 0.9830044252696218
Recall: 0.9828824723807696
F1: 0.9829195656898114


In [114]:
import os
import json

# Adversarial test function
def adv_test(eps: int):
    print(f"=== Eps: {eps} ===")
    attack = FastGradientMethod(estimator=classifier, eps=eps)
    
    x_test_adv = attack.generate(x=x_test_norm)#, y=y_test[:fraction])
    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == y_test) / pred_classes.size
    print(f"Adversarial accuracy: {adversarial_accuracy}")

    true_flat = y_test.flatten()
    pred_flat = pred_classes.flatten()
    precision = precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    # Save metrics to JSON in ./output
    os.makedirs("./output", exist_ok=True)
    metrics = {
        "eps": eps,
        "accuracy": float(adversarial_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    output_path = f"./output/adv_test_eps_{eps}.json"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return metrics

In [120]:
# epsilon sweep for FGSM
for i in range(3, 31, 1):
    eps = float(i/100)
    adv_test(eps)


=== Eps: 0.03 ===
Adversarial accuracy: 0.9741106687770796
Precision: 0.9751466851740929
Recall: 0.9741106687770796
F1: 0.9743102784528171
Saved metrics to ./output/adv_test_eps_0.03.json
=== Eps: 0.04 ===
Adversarial accuracy: 0.9735862631301262
Precision: 0.9748306327082545
Recall: 0.9735862631301262
F1: 0.9738123041531938
Saved metrics to ./output/adv_test_eps_0.04.json
=== Eps: 0.05 ===
Adversarial accuracy: 0.9532031912002175
Precision: 0.9585186457884973
Recall: 0.9532031912002175
F1: 0.9540232321065347
Saved metrics to ./output/adv_test_eps_0.05.json
=== Eps: 0.06 ===
Adversarial accuracy: 0.846638527827074
Precision: 0.8976465008429665
Recall: 0.846638527827074
F1: 0.8529321716365843
Saved metrics to ./output/adv_test_eps_0.06.json
=== Eps: 0.07 ===
Adversarial accuracy: 0.7676635170351895
Precision: 0.8688139185556494
Recall: 0.7676635170351895
F1: 0.7772963918875551
Saved metrics to ./output/adv_test_eps_0.07.json
=== Eps: 0.08 ===
Adversarial accuracy: 0.7470574128255552
Pre

Normalized:
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved backup of main model.
SAVE PATH: out/FL/RandPos-Normalized-False-20-30-30-200/mainModelBackup.ckpt
Elapsed time (ns): 20533253865914